In [5]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import date
from pathlib import Path
from lifelines import LogNormalAFTFitter
from lifelines.utils import k_fold_cross_validation
from sklearn.model_selection import train_test_split
from src.config import Paths, ModelConfig

In [6]:
df_full = pd.read_csv(Paths.encoded_data)

df_train, df_holdout = train_test_split(df_full, test_size=0.2, random_state=42)

df_holdout.to_csv(Paths.encoded_data.parent.parent / 'test' / 'holdout_set.csv', index=False)

print(f"Train: {len(df_train)} | Holdout: {len(df_holdout)}")
df_train.head()

Train: 5634 | Holdout: 1409


,Age,Number_of_Dependents,Number_of_Referrals,Tenure_in_Months,Avg_Monthly_Long_Distance_Charges,Avg_Monthly_GB_Download,Num_Internet_Features,Has_Multiple_Lines,Churn_status,gender_e,married_e,paper_e,offer_e,Contract_e,Payment_Method_e,Internet_Type_DSL,Internet_Type_Fiber Optic
2142,38,2,1,3,33.30,12.0,5,1,0,1,1,0,1,0,0,0,1
1623,22,0,0,36,43.97,42.0,4,2,1,1,0,1,1,0,0,0,1
6074,53,0,2,49,49.63,9.0,5,2,1,0,1,1,0,0,0,0,1
1362,54,0,0,7,19.21,16.0,3,1,1,0,0,1,0,0,0,0,1
6754,38,0,8,2,15.50,12.0,2,1,0,0,1,1,1,0,0,1,0


In [7]:
# drop skewed columns 
COLS_TO_DROP = ModelConfig.cols_to_drop
DURATION_COL = ModelConfig.duration_col
EVENT_COL    = ModelConfig.event_col

df = df_train.drop(columns=[c for c in COLS_TO_DROP if c in df_train.columns])

print("Remaining columns:", df.columns.tolist())

Remaining columns: ['Age', 'Tenure_in_Months', 'Churn_status', 'gender_e', 'married_e', 'paper_e', 'offer_e', 'Contract_e', 'Payment_Method_e', 'Internet_Type_Fiber Optic']


In [8]:
# train AFT model and validation using 5-fold cross validation

aft = LogNormalAFTFitter()

cv_scores = k_fold_cross_validation(
    aft, df,
    duration_col=DURATION_COL,
    event_col=EVENT_COL,
    k=5,
    scoring_method="concordance_index"
)

c_index = float(np.mean(cv_scores))
print(f"CV Concordance Index (mean): {c_index:.3f}")
print(f"Individual fold scores: {np.round(cv_scores, 3)}")

CV Concordance Index (mean): 0.837
Individual fold scores: [0.841 0.848 0.831 0.84  0.827]


In [9]:
# fit on the full dataset and print summary 

aft.fit(df, duration_col=DURATION_COL, event_col=EVENT_COL)
aft.print_summary()

<lifelines.LogNormalAFTFitter: fitted with 5634 total observations, 4138 right-censored observations>
             duration col = 'Tenure_in_Months'
                event col = 'Churn_status'
   number of observations = 5634
number of events observed = 1496
           log-likelihood = -7152.68
         time fit was run = 2026-04-19 21:06:35 UTC

---
                                  coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
param  covariate                                                                                                                  
mu_    Age                       -0.01      0.99      0.00           -0.01           -0.00                0.99                1.00
       Contract_e                 3.24     25.62      0.09            3.07            3.41               21.59               30.41
       Internet_Type_Fiber Optic -0.21      0.81      0.07           -0.35           -0.07                0.71                0.93
       Payment_Method_e           0.61      1.84      0.08            0.46            0.76                1.59                2.14
       gender_e                   0.01      1.01      0.07           -0.12            0.14                0.89                1.15
       married_e                  0.84      2.31      0.07            0.70            0.97                2.02                2.65
       offer_e                   -0.01      0.99      0.07           -0.14            0.12                0.87                1.13
       paper_e                   -0.09      0.91      0.07           -0.24            0.06                0.79                1.06
       Intercept                  3.20     24.48      0.13            2.95            3.45               19.06               31.44
sigma_ Intercept                  0.54      1.72      0.02            0.50            0.58                1.65                1.78

                                  cmp to     z      p  -log2(p)
param  covariate                                               
mu_    Age                          0.00 -3.92 <0.005     13.44
       Contract_e                   0.00 37.10 <0.005    998.34
       Internet_Type_Fiber Optic    0.00 -2.92 <0.005      8.14
       Payment_Method_e             0.00  8.04 <0.005     49.99
       gender_e                     0.00  0.13   0.89      0.16
       married_e                    0.00 12.16 <0.005    110.57
       offer_e                      0.00 -0.10   0.92      0.12
       paper_e                      0.00 -1.21   0.23      2.13
       Intercept                    0.00 25.05 <0.005    457.52
sigma_ Intercept                    0.00 28.03 <0.005    571.71
---
Concordance = 0.84
AIC = 14325.35
log-likelihood ratio test = 2500.76 on 8 df
-log2(p) of ll-ratio test = inf

In [11]:
# save the model and metadata

Paths.model.parent.mkdir(exist_ok=True)

joblib.dump(aft, Paths.model)

metadata = {
    "model": "LogNormalAFT",
    "training_date": str(date.today()),
    "c_index_cv_mean": round(c_index, 4),
    "cv_folds": 5,
    "duration_col": DURATION_COL,
    "event_col": EVENT_COL,
    "dropped_columns": COLS_TO_DROP,
    "training_rows": len(df),
    "holdout_rows": len(df_holdout),
    "train_test_split": "80/20",
}

with open(Paths.model_metadata, 'w') as f:
    json.dump(metadata, f, indent=4)

print("Model and metadata saved successfully.")

Model and metadata saved successfully.
